In [3]:
import numpy as np 
import pandas as pd 
import torch 
import torch.nn as nn 
from torch.utils.data import DataLoader,TensorDataset
from sklearn.preprocessing import MinMaxScaler
import yfinance as download_yf

1: Tai du lieu tu Yahoo Finnace (Vi du ma co phieu :AAPL)


In [4]:
df=download_yf.download('AAPL',start='2020-01-01',end='2026-01-01')
data=df['Close'].values.reshape(-1,1)

[*********************100%***********************]  1 of 1 completed


2 : Chuan hoa ve du lieu 

In [5]:
scaler=MinMaxScaler(feature_range=(0,1))
scaled_data=scaler.fit_transform(data)

3:Tao ham tao chuoi 

In [6]:
def create_sequences(data,seq_length):
    X,y=[],[]
    for i in range(len(data)-seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)
SEQ_LENGTH=60
X,y=create_sequences(scaled_data,SEQ_LENGTH)
# Reshape thanh (batch_size, seq_length, num_features)
X=torch.tensor(X,dtype=torch.float32).unsqueeze(-1)
y=torch.tensor(y,dtype=torch.float32).unsqueeze(-1)
#Chia train/test(80% train, 20% test)
train_size=int(len(X)*0.8)
X_train,y_train=X[:train_size],y[:train_size]
X_test,y_test=X[train_size:],y[train_size:]
train_loader=DataLoader(TensorDataset(X_train,y_train),batch_size=32,shuffle=True)
test_loader=DataLoader(TensorDataset(X_test,y_test),batch_size=32,shuffle=False)

4: Dinh nghia mo hinh LSTM 

In [7]:
class LSTMModel(nn.Module):
    def __init__(self,input_size,hidden_size,num_layers,output_size):
        super(LSTMModel,self).__init__()
        self.hidden_size=hidden_size
        self.num_layers=num_layers
        self.lstm=nn.LSTM(input_size,hidden_size,num_layers,batch_first=True)
        self.fc=nn.Linear(hidden_size,output_size)
    def forward(self,x):
        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size).to(x.device)
        c0=torch.zeros(self.num_layers,x.size(0),self.hidden_size).to(x.device)
        out,_=self.lstm(x,(h0,c0))
        out=self.fc(out[:,-1,:])
        return out
model=LSTMModel(input_size=1,hidden_size=50,num_layers=2,output_size=1)
criterion=nn.MSELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)    

5: Huan luyen mo hinh (Training lood ngan gon)


In [11]:
model.train()
for epoch in range(20):
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        batch_X = batch_X.squeeze(-1)  # (batch, sequence, features)
        batch_y = batch_y.squeeze(-1)  # match model output shape
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    print(f"Epoch [{epoch+1}/20], Loss: {loss.item():.4f}")

Epoch [1/20], Loss: 0.0528
Epoch [2/20], Loss: 0.0047
Epoch [3/20], Loss: 0.0040
Epoch [4/20], Loss: 0.0004
Epoch [5/20], Loss: 0.0008
Epoch [6/20], Loss: 0.0010
Epoch [7/20], Loss: 0.0007
Epoch [8/20], Loss: 0.0008
Epoch [9/20], Loss: 0.0007
Epoch [10/20], Loss: 0.0000
Epoch [11/20], Loss: 0.0001
Epoch [12/20], Loss: 0.0002
Epoch [13/20], Loss: 0.0002
Epoch [14/20], Loss: 0.0008
Epoch [15/20], Loss: 0.0003
Epoch [16/20], Loss: 0.0010
Epoch [17/20], Loss: 0.0007
Epoch [18/20], Loss: 0.0012
Epoch [19/20], Loss: 0.0001
Epoch [20/20], Loss: 0.0001
